# A/B Testing with Chi-Square Test of Independence

## Notebook 3: Chi-Square Tests for Categorical Outcomes

**Purpose**: Use chi-square tests to determine if there's a statistically significant relationship between treatment group and categorical outcomes (visit, conversion).

### What is the Chi-Square Test?

The chi-square test examines whether **observed frequencies** in data match **expected frequencies** under the null hypothesis of independence.

**Core Question**: Is there an association between treatment group and outcome category, or are they independent?

- **H₀ (Null)**: Treatment and outcome are independent (no association)
- **H₁ (Alternative)**: Treatment and outcome are related (association exists)

### When to Use Chi-Square?

- Both variables are categorical
- Sample sizes are reasonably large (expected frequencies > 5 in most cells)
- Observations are independent
- It's a test of association/independence, not causation

This notebook covers chi-square tests for our binary outcomes: visit (yes/no) and conversion (yes/no).

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency
from scipy.stats import chi2
import warnings
warnings.filterwarnings('ignore')

# Try to import plotly
try:
    import plotly.express as px
    import plotly.graph_objects as go
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False
    print("Plotly not available - static plots will be used")

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

# Create output directory
os.makedirs('../data/outputs/nb03', exist_ok=True)


In [ ]:
# Load cleaned dataset
df = pd.read_csv('../data/outputs/nb01/nb01_hillstrom_clean.csv')

print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nTreatment groups: {df['segment'].unique()}")
print(f"\nOutcome variables: visit, conversion")

## Chi-Square Test Basics

### Contingency Tables

A **contingency table** (or cross-tabulation) shows the joint distribution of two categorical variables.

**Example**: Visit × Treatment Group

|  | Mens Email | Womens Email | No Email | Total |
|---|---|---|---|---|
| No Visit (0) | 7,500 | 7,600 | 7,900 | 23,000 |
| Visit (1) | 8,500 | 8,400 | 8,100 | 25,000 |
| **Total** | **16,000** | **16,000** | **16,000** | **48,000** |

### Chi-Square Statistic

$$\chi^2 = \sum \frac{(O - E)^2}{E}$$

Where:
- O = observed frequency (actual data)
- E = expected frequency (what we'd expect if H₀ is true)

**Interpretation**:
- Large χ²: Observed differs greatly from expected → reject H₀
- Small χ²: Observed matches expected → fail to reject H₀

### Degrees of Freedom

$$df = (r - 1)(c - 1)$$

Where r = number of rows, c = number of columns

For a 2×3 table (2 outcomes × 3 groups): df = (2-1)(3-1) = 2

## Step 1: Build Contingency Tables

Let's create contingency tables for our two binary outcomes.

In [ ]:
# Contingency table for VISIT
print("=" * 70)
print("CONTINGENCY TABLE: VISIT by TREATMENT GROUP")
print("=" * 70)

visit_table = pd.crosstab(df['segment'], df['visit'], margins=True)
print(visit_table)

# Add proportions
print("\nProportion within each group:")
visit_props = pd.crosstab(df['segment'], df['visit'], normalize='index')
print(visit_props.round(4))

In [ ]:
# Contingency table for CONVERSION
print("\n" + "=" * 70)
print("CONTINGENCY TABLE: CONVERSION by TREATMENT GROUP")
print("=" * 70)

conv_table = pd.crosstab(df['segment'], df['conversion'], margins=True)
print(conv_table)

# Add proportions
print("\nProportion within each group:")
conv_props = pd.crosstab(df['segment'], df['conversion'], normalize='index')
print(conv_props.round(4))

## Step 2: Perform Chi-Square Test of Independence

For each outcome (visit and conversion), we'll test whether treatment group and outcome are independent using the chi-square test.

In [ ]:
def perform_chi_square_test(contingency_table, outcome_name):
    """Perform chi-square test on a contingency table."""
    # Remove margins if present
    if 'All' in contingency_table.columns:
        ct = contingency_table.drop('All', axis=1)
    else:
        ct = contingency_table
    if 'All' in ct.index:
        ct = ct.drop('All', axis=0)
    else:
        ct = ct
    
    # Perform chi-square test
    chi2_stat, p_val, dof, expected_freq = chi2_contingency(ct)
    
    return {
        'outcome': outcome_name,
        'chi2': chi2_stat,
        'p_value': p_val,
        'dof': dof,
        'expected_freq': expected_freq,
        'contingency_table': ct
    }

# Test for VISIT
visit_ct = pd.crosstab(df['segment'], df['visit'])
visit_test = perform_chi_square_test(visit_ct, 'Visit')

print("=" * 70)
print("CHI-SQUARE TEST: TREATMENT × VISIT")
print("=" * 70)
print(f"\nNull Hypothesis (H₀): Treatment group and visit are independent")
print(f"Alternative Hypothesis (H₁): Treatment group and visit are related")
print(f"\nContingency Table:")
print(visit_ct)
print(f"\nChi-square statistic: {visit_test['chi2']:.4f}")
print(f"P-value: {visit_test['p_value']:.10f}")
print(f"Degrees of freedom: {visit_test['dof']}")
print(f"\nResult: {'REJECT H₀' if visit_test['p_value'] < 0.05 else 'FAIL TO REJECT H₀'} (α = 0.05)")
if visit_test['p_value'] < 0.05:
    print(f"Conclusion: Significant association between treatment and visit (p < 0.05)")
else:
    print(f"Conclusion: No significant association (p ≥ 0.05)")

In [ ]:
# Test for CONVERSION
conv_ct = pd.crosstab(df['segment'], df['conversion'])
conv_test = perform_chi_square_test(conv_ct, 'Conversion')

print("\n" + "=" * 70)
print("CHI-SQUARE TEST: TREATMENT × CONVERSION")
print("=" * 70)
print(f"\nNull Hypothesis (H₀): Treatment group and conversion are independent")
print(f"Alternative Hypothesis (H₁): Treatment group and conversion are related")
print(f"\nContingency Table:")
print(conv_ct)
print(f"\nChi-square statistic: {conv_test['chi2']:.4f}")
print(f"P-value: {conv_test['p_value']:.10f}")
print(f"Degrees of freedom: {conv_test['dof']}")
print(f"\nResult: {'REJECT H₀' if conv_test['p_value'] < 0.05 else 'FAIL TO REJECT H₀'} (α = 0.05)")
if conv_test['p_value'] < 0.05:
    print(f"Conclusion: Significant association between treatment and conversion (p < 0.05)")
else:
    print(f"Conclusion: No significant association (p ≥ 0.05)")

## Step 3: Check Chi-Square Assumptions

### Assumption 1: Expected Frequency > 5

For chi-square to be valid, most cells should have expected frequency ≥ 5. Let's verify this.

In [ ]:
print("=" * 70)
print("EXPECTED FREQUENCIES: VISIT")
print("=" * 70)
expected_visit = visit_test['expected_freq']
print(expected_visit)
print(f"\nMinimum expected frequency: {expected_visit.min():.2f}")
print(f"Cells with expected frequency < 5: {(expected_visit < 5).sum()}")
print(f"Valid for chi-square? {'✓ Yes' if (expected_visit < 5).sum() == 0 else '✗ No'}")

print("\n" + "=" * 70)
print("EXPECTED FREQUENCIES: CONVERSION")
print("=" * 70)
expected_conv = conv_test['expected_freq']
print(expected_conv)
print(f"\nMinimum expected frequency: {expected_conv.min():.2f}")
print(f"Cells with expected frequency < 5: {(expected_conv < 5).sum()}")
print(f"Valid for chi-square? {'✓ Yes' if (expected_conv < 5).sum() == 0 else '✗ No'}")

## Step 4: Pairwise Chi-Square Tests

The overall test shows whether treatment is associated with outcome. Now let's perform pairwise comparisons between specific groups using chi-square tests.

**Comparisons**:
1. Mens Email vs Control
2. Womens Email vs Control
3. Mens Email vs Womens Email

In [ ]:
def chi_square_2group(data, seg1, seg2, outcome):
    """Perform chi-square test for two groups."""
    d1 = data[data['segment'] == seg1][outcome]
    d2 = data[data['segment'] == seg2][outcome]
    
    ct = pd.crosstab([d1.index], d1)
    ct2 = pd.crosstab([d2.index], d2)
    
    combined_ct = pd.concat([
        pd.Series([d1.value_counts().get(0, 0), d1.value_counts().get(1, 0)]),
        pd.Series([d2.value_counts().get(0, 0), d2.value_counts().get(1, 0)])
    ], axis=1).T
    
    chi2, p_val, dof, expected = chi2_contingency(combined_ct)
    return chi2, p_val

print("=" * 70)
print("PAIRWISE CHI-SQUARE TESTS: CONVERSION")
print("=" * 70)

segments = ["Mens E-Mail", "Womens E-Mail", "No E-Mail"]
pairwise_results = []

# Men vs Control
chi2_mc, p_mc = chi_square_2group(df, "Mens E-Mail", "No E-Mail", 'conversion')
pairwise_results.append({
    'Comparison': "Mens Email vs Control",
    'Chi-Square': chi2_mc,
    'P-value': p_mc,
    'Significant': 'Yes' if p_mc < 0.05 else 'No'
})
print(f"Mens Email vs Control: χ² = {chi2_mc:.4f}, p = {p_mc:.6f}")

# Women vs Control
chi2_wc, p_wc = chi_square_2group(df, "Womens E-Mail", "No E-Mail", 'conversion')
pairwise_results.append({
    'Comparison': "Womens Email vs Control",
    'Chi-Square': chi2_wc,
    'P-value': p_wc,
    'Significant': 'Yes' if p_wc < 0.05 else 'No'
})
print(f"Womens Email vs Control: χ² = {chi2_wc:.4f}, p = {p_wc:.6f}")

# Men vs Women
chi2_mw, p_mw = chi_square_2group(df, "Mens E-Mail", "Womens E-Mail", 'conversion')
pairwise_results.append({
    'Comparison': "Mens Email vs Womens Email",
    'Chi-Square': chi2_mw,
    'P-value': p_mw,
    'Significant': 'Yes' if p_mw < 0.05 else 'No'
})
print(f"Mens Email vs Womens Email: χ² = {chi2_mw:.4f}, p = {p_mw:.6f}")

In [ ]:
# Apply Bonferroni correction to pairwise tests
num_pairwise = 3
alpha_bonf = 0.05 / num_pairwise

print("\n" + "=" * 70)
print("BONFERRONI CORRECTION FOR PAIRWISE TESTS")
print("=" * 70)
print(f"Number of pairwise comparisons: {num_pairwise}")
print(f"Adjusted α: {alpha_bonf:.6f}\n")

pw_df = pd.DataFrame(pairwise_results)
pw_df['Significant (Bonf)'] = pw_df['P-value'] < alpha_bonf
print(pw_df.to_string(index=False))

## Step 5: Effect Size - Cramér's V

While chi-square tells us if an association exists, **Cramér's V** quantifies the strength of the association.

### Formula

$$V = \sqrt{\frac{\chi^2}{n \times \min(k-1, r-1)}}$$

Where:
- χ² = chi-square statistic
- n = sample size
- k = number of columns
- r = number of rows

### Interpretation

- V = 0: No association
- 0 < V < 0.1: Very weak association
- 0.1 ≤ V < 0.3: Weak association
- 0.3 ≤ V < 0.5: Moderate association
- V ≥ 0.5: Strong association

In [ ]:
def cramers_v(chi2_stat, n, min_dim):
    """Calculate Cramér's V effect size."""
    return np.sqrt(chi2_stat / (n * min_dim))

print("=" * 70)
print("CRAMÉR'S V: EFFECT SIZE FOR CHI-SQUARE")
print("=" * 70)

# For Visit
n = len(df)
min_dim_visit = min(visit_ct.shape[0] - 1, visit_ct.shape[1] - 1)
v_visit = cramers_v(visit_test['chi2'], n, min_dim_visit)

print(f"\nVISIT Outcome:")
print(f"  Cramér's V: {v_visit:.4f}")
print(f"  Interpretation: {'Very weak' if v_visit < 0.1 else 'Weak' if v_visit < 0.3 else 'Moderate' if v_visit < 0.5 else 'Strong'} association")

# For Conversion
min_dim_conv = min(conv_ct.shape[0] - 1, conv_ct.shape[1] - 1)
v_conv = cramers_v(conv_test['chi2'], n, min_dim_conv)

print(f"\nCONVERSION Outcome:")
print(f"  Cramér's V: {v_conv:.4f}")
print(f"  Interpretation: {'Very weak' if v_conv < 0.1 else 'Weak' if v_conv < 0.3 else 'Moderate' if v_conv < 0.5 else 'Strong'} association")

## Step 6: Residual Analysis

Chi-square is an omnibus test - it tells us if association exists, but not where the differences lie.

**Standardized Residuals** show which cells contribute most to the chi-square statistic:

$$\text{Residual} = \frac{O - E}{\sqrt{E}}$$

Large residuals (|residual| > 2) indicate cells that differ significantly from expected.

In [ ]:
def calculate_standardized_residuals(contingency_table):
    """Calculate standardized residuals for contingency table."""
    chi2_stat, p_val, dof, expected = chi2_contingency(contingency_table)
    standardized_resid = (contingency_table.values - expected) / np.sqrt(expected)
    return pd.DataFrame(standardized_resid, 
                       index=contingency_table.index, 
                       columns=contingency_table.columns)

print("=" * 70)
print("STANDARDIZED RESIDUALS: VISIT")
print("=" * 70)
resid_visit = calculate_standardized_residuals(visit_ct)
print(resid_visit.round(3))
print("\nLarge residuals (|value| > 2) indicate cells contributing most to association")

print("\n" + "=" * 70)
print("STANDARDIZED RESIDUALS: CONVERSION")
print("=" * 70)
resid_conv = calculate_standardized_residuals(conv_ct)
print(resid_conv.round(3))
print("\nLarge residuals (|value| > 2) indicate cells contributing most to association")

In [ ]:
# Heatmap of standardized residuals
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals for Visit
sns.heatmap(resid_visit, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            cbar_kws={'label': 'Standardized Residual'}, ax=axes[0], 
            vmin=-3, vmax=3, linewidths=1, linecolor='black')
axes[0].set_title('Standardized Residuals: Visit × Treatment', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Visit Status', fontsize=10)
axes[0].set_xlabel('Treatment Group', fontsize=10)

# Residuals for Conversion
sns.heatmap(resid_conv, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            cbar_kws={'label': 'Standardized Residual'}, ax=axes[1],
            vmin=-3, vmax=3, linewidths=1, linecolor='black')
axes[1].set_title('Standardized Residuals: Conversion × Treatment', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Conversion Status', fontsize=10)
axes[1].set_xlabel('Treatment Group', fontsize=10)

plt.tight_layout()
plt.savefig('../data/outputs/nb03/nb03_chi_square_residuals.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: ../data/outputs/nb03/nb03_chi_square_residuals.png")

In [ ]:
# Visualize proportions with stacked bar charts
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Visit proportions
visit_props = pd.crosstab(df['segment'], df['visit'], normalize='index') * 100
visit_props.plot(kind='bar', stacked=True, ax=axes[0], 
                 color=['#d62728', '#2ca02c'], alpha=0.8, edgecolor='black', linewidth=1)
axes[0].set_title('Visit Rates by Treatment Group', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Percentage (%)', fontsize=10)
axes[0].set_xlabel('Treatment Group', fontsize=10)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45)
axes[0].legend(['No Visit', 'Visit'], title='Outcome')
axes[0].grid(axis='y', alpha=0.3)

# Conversion proportions
conv_props = pd.crosstab(df['segment'], df['conversion'], normalize='index') * 100
conv_props.plot(kind='bar', stacked=True, ax=axes[1],
                color=['#d62728', '#2ca02c'], alpha=0.8, edgecolor='black', linewidth=1)
axes[1].set_title('Conversion Rates by Treatment Group', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Percentage (%)', fontsize=10)
axes[1].set_xlabel('Treatment Group', fontsize=10)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)
axes[1].legend(['No Conversion', 'Conversion'], title='Outcome')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../data/outputs/nb03/nb03_chi_square_proportions.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: ../data/outputs/nb03/nb03_chi_square_proportions.png")

In [ ]:
# Create a visual representation of expected vs observed
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Observed frequencies - Visit
obs_visit = visit_ct.T
sns.heatmap(obs_visit, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            cbar_kws={'label': 'Observed Frequency'}, linewidths=1, linecolor='black')
axes[0].set_title('Observed Frequencies: Visit × Treatment', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Treatment Group', fontsize=10)
axes[0].set_ylabel('Visit Status', fontsize=10)

# Observed frequencies - Conversion
obs_conv = conv_ct.T
sns.heatmap(obs_conv, annot=True, fmt='d', cmap='Oranges', ax=axes[1],
            cbar_kws={'label': 'Observed Frequency'}, linewidths=1, linecolor='black')
axes[1].set_title('Observed Frequencies: Conversion × Treatment', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Treatment Group', fontsize=10)
axes[1].set_ylabel('Conversion Status', fontsize=10)

plt.tight_layout()
plt.savefig('../data/outputs/nb03/nb03_chi_square_observed.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Saved: ../data/outputs/nb03/nb03_chi_square_observed.png")

## Step 7: Summary of Chi-Square Results

Let's compile all chi-square test results into a comprehensive summary.

In [ ]:
# Comprehensive chi-square results summary
chi_results = []

# Overall tests
chi_results.append({
    'Test': 'Overall: Treatment × Visit',
    'Chi-Square': f"{visit_test['chi2']:.4f}",
    'DF': visit_test['dof'],
    'P-value': f"{visit_test['p_value']:.6f}",
    'Significant (α=0.05)': 'Yes' if visit_test['p_value'] < 0.05 else 'No',
    "Cramér's V": f"{v_visit:.4f}",
    'Effect': 'Very weak' if v_visit < 0.1 else 'Weak'
})

chi_results.append({
    'Test': 'Overall: Treatment × Conversion',
    'Chi-Square': f"{conv_test['chi2']:.4f}",
    'DF': conv_test['dof'],
    'P-value': f"{conv_test['p_value']:.6f}",
    'Significant (α=0.05)': 'Yes' if conv_test['p_value'] < 0.05 else 'No',
    "Cramér's V": f"{v_conv:.4f}",
    'Effect': 'Very weak' if v_conv < 0.1 else 'Weak'
})

chi_df = pd.DataFrame(chi_results)
print("=" * 70)
print("CHI-SQUARE TEST SUMMARY")
print("=" * 70)
print(chi_df.to_string(index=False))

# Save results
chi_df.to_csv('../data/outputs/nb03/nb03_chi_square_results.csv', index=False)
print("\n✓ Saved: ../data/outputs/nb03/nb03_chi_square_results.csv")

## Step 8: Chi-Square Assumptions Summary

### Assumption 1: Expected Frequencies
- ✓ All expected frequencies > 5 (verified above)
- Large sample size ensures stability of chi-square test

### Assumption 2: Independence of Observations
- ✓ Random assignment ensures independence between groups
- ✓ Each customer appears only once in dataset

### Assumption 3: Sample Size
- ✓ Sample size (n ≈ 64,000) is very large
- Large samples increase statistical power to detect true effects

### Conclusion on Assumptions
All chi-square assumptions are satisfied. Tests results are valid and reliable.

## How to Interpret Chi-Square Results

### If P-value < 0.05 (Reject H₀):
- **Finding**: Treatment group and outcome are statistically associated
- **Meaning**: Email campaign (treatment) is related to visit/conversion rates
- **Action**: Examine effect size (Cramér's V) to assess practical significance
  - Small effect (V < 0.1): Association exists but is weak
  - Large effect (V ≥ 0.3): Association is strong and practically important

### If P-value ≥ 0.05 (Fail to Reject H₀):
- **Finding**: No statistically significant association detected
- **Meaning**: Evidence insufficient to conclude treatment affects outcomes
- **Caution**: Absence of evidence ≠ evidence of absence
  - Could be underpowered (small effect size)
  - Could be no real effect

### Effect Size Interpretation (Cramér's V):
- **Very Weak** (V < 0.1): Minimal practical difference
- **Weak** (0.1 ≤ V < 0.3): Small but noticeable difference
- **Moderate** (0.3 ≤ V < 0.5): Substantial difference
- **Strong** (V ≥ 0.5): Large, important difference

## Key Takeaways

1. **Chi-Square Tests**:
   - Test for association between categorical variables
   - Compare observed vs. expected frequencies under independence
   - Produce chi-square statistic and p-value
   - Omnibus test - doesn't show which groups differ

2. **Contingency Tables**:
   - Show joint distribution of two categorical variables
   - Foundation for chi-square calculations
   - Proportions reveal pattern differences between groups

3. **Pairwise Tests**:
   - Follow-up comparisons after significant overall test
   - Use Bonferroni correction to control false positives
   - Identify which group pairs differ significantly

4. **Effect Size (Cramér's V)**:
   - Quantifies strength of association (0 to 1)
   - Statistical significance ≠ practical significance
   - Small effect: p < 0.05 but V < 0.1 (weak practical importance)

5. **Residual Analysis**:
   - Standardized residuals show which cells drive the association
   - Large residuals (|value| > 2) = cells differing from expectation
   - Helps identify which outcomes/groups differ most

6. **Assumptions**:
   - Expected frequencies > 5 in most cells
   - Independent observations (satisfied by random assignment)
   - Large sample sizes provide stable estimates

---

## Comprehensive A/B Testing Summary

Across three notebooks, we've covered:

1. **Notebook 1 - EDA**: Explore data, assess balance, understand outcomes
2. **Notebook 2 - Frequentist Tests**: Z-tests and t-tests for proportions and means
3. **Notebook 3 - Chi-Square**: Test categorical associations, effect sizes, residuals

Together, these provide a complete frequentist framework for A/B testing analysis.